# Tai Dinh, Week 3, Data Preprocessing

In [169]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [170]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import files

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Loading data
Loading 6 months of data and combining into one dataframe

In [171]:
jan24 = pd.read_csv('/content/drive/MyDrive/IDX_data/CRMLSSold202401_filled.csv')
feb24 = pd.read_csv('/content/drive/MyDrive/IDX_data/CRMLSSold202402_filled.csv')
mar25 = pd.read_csv('/content/drive/MyDrive/IDX_data/CRMLSSold202503.csv')
april25 = pd.read_csv('/content/drive/MyDrive/IDX_data/CRMLSSold202504.csv')
april26 = pd.read_csv('/content/drive/MyDrive/IDX_data/CRMLSSold202604.csv')
may26 = pd.read_csv('/content/drive/MyDrive/IDX_data/CRMLSSold202605.csv')

In [172]:
dfs = [jan24, feb24, mar25, april25, april26, may26]
df = pd.concat(dfs, ignore_index=True)
df.shape

(129262, 82)

In [173]:
df = df.drop(columns = ["BuyerAgentAOR", "ListAgentAOR"])
df.head()

,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,latfilled,lonfilled
0,"Carpet,Tile,Wood",True,NaN,NaN,False,499000.0,551985747,jwachter@cbnorcal.com,2024-01-26,240000.0,...,NaN,False,1.0,Other,94401,6472.0,NaN,NaN,True,True
1,NaN,NaN,NaN,NaN,NaN,0.0,535486633,eabrown@lee-associates.com,2024-01-24,950.0,...,NaN,NaN,NaN,NaN,92394,NaN,52320.0,NaN,False,False
2,NaN,True,NaN,NaN,NaN,75000.0,529986282,Joe@9WINWIN.com,2024-01-16,45000.0,...,NaN,False,NaN,NaN,93240,NaN,217364.0,NaN,False,False
3,NaN,True,NaN,NaN,NaN,199000.0,529618166,carolthefinder@yahoo.com,2024-01-08,141500.0,...,NaN,False,NaN,NaN,92308,NaN,217800.0,NaN,False,False
4,NaN,True,NaN,NaN,NaN,19500.0,522614340,jtavisola@tavisola.com,2024-01-17,15000.0,...,NaN,False,NaN,NaN,93544,0.0,108883.0,NaN,False,False


In [174]:
#focus on residential single family homes
df = df[(df["PropertyType"] == "Residential") & (df["PropertySubType"] == "SingleFamilyResidence")].copy()

In [175]:
#shape and info on filtered dataset
print(df.shape)
df.info()

(64431, 80)
<class 'pandas.core.frame.DataFrame'>
Index: 64431 entries, 5 to 129258
Data columns (total 80 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Flooring                      40920 non-null  object 
 1   ViewYN                        58727 non-null  object 
 2   WaterfrontYN                  37 non-null     object 
 3   BasementYN                    1536 non-null   object 
 4   PoolPrivateYN                 59017 non-null  object 
 5   OriginalListPrice             64287 non-null  float64
 6   ListingKey                    64431 non-null  int64  
 7   ListAgentEmail                64291 non-null  object 
 8   CloseDate                     64431 non-null  object 
 9   ClosePrice                    64431 non-null  float64
 10  ListAgentFirstName            63993 non-null  object 
 11  ListAgentLastName             64423 non-null  object 
 12  Latitude                      64406 non-null  float6

In [176]:
df.describe()

,OriginalListPrice,ListingKey,ClosePrice,Latitude,Longitude,LivingArea,ListPrice,DaysOnMarket,FireplacesTotal,AboveGradeFinishedArea,...,ElementarySchoolDistrict,BelowGradeFinishedArea,CoveredSpaces,Stories,LotSizeArea,MainLevelBedrooms,GarageSpaces,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict
count,6.428700e+04,6.443100e+04,6.443100e+04,64406.000000,64406.000000,64397.000000,6.443100e+04,64431.000000,0.0,0.0,...,0.0,424.000000,0.0,56909.000000,6.326700e+04,38781.000000,62064.000000,45563.000000,6.326500e+04,0.0
mean,1.340106e+06,1.107050e+09,1.302748e+06,34.729088,-118.592623,2036.838115,1.250289e+06,35.909500,NaN,NaN,...,NaN,67.518868,NaN,1.349207,2.295147e+04,2.260359,1.999408,106.578195,1.103074e+05,NaN
std,7.033405e+06,4.212100e+07,6.488464e+06,1.777307,3.320195,1017.571542,1.495785e+06,51.359574,NaN,NaN,...,NaN,274.287712,NaN,0.476724,9.402040e+05,1.427771,3.628869,350.985442,9.254982e+06,NaN
min,1.000000e+01,4.216785e+08,6.850000e+02,-22.863239,-124.144091,0.000000,1.999900e+04,-39.000000,NaN,NaN,...,NaN,0.000000,NaN,1.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,NaN
25%,6.249000e+05,1.058414e+09,6.150000e+05,33.751801,-119.172591,1378.000000,6.150000e+05,7.000000,NaN,NaN,...,NaN,0.000000,NaN,1.000000,5.446000e+03,1.000000,2.000000,0.000000,5.663000e+03,NaN
50%,8.899000e+05,1.107352e+09,8.850000e+05,34.076981,-118.014789,1810.000000,8.790000e+05,16.000000,NaN,NaN,...,NaN,0.000000,NaN,1.000000,7.100000e+03,3.000000,2.000000,0.000000,7.253000e+03,NaN
75%,1.400000e+06,1.153268e+09,1.425000e+06,34.862058,-117.251033,2430.000000,1.399000e+06,44.000000,NaN,NaN,...,NaN,0.000000,NaN,2.000000,9.927000e+03,3.000000,2.000000,135.000000,1.038500e+04,NaN
max,9.490000e+08,1.171669e+09,9.700000e+08,41.878351,329.000000,23314.000000,6.995000e+07,966.000000,NaN,NaN,...,NaN,2941.000000,NaN,2.000000,2.178000e+08,44.000000,600.000000,17220.000000,2.087221e+09,NaN


# Handling Missing or Incorrect Values

In [177]:
#drop properties with 0 living area, not valid data
df = df[df["LivingArea"] > 0].reset_index(drop=True)
df.shape

#impute missing numeric values in specified cols with median(resistant to high outliers in dataset)
med_imputer = SimpleImputer(strategy='median')

#impute certain numeric columns with 0/1 if not there
zero_imputer = SimpleImputer(strategy='constant', fill_value=0)
one_imputer = SimpleImputer(strategy = "constant", fill_value = 1)

#impute boolean columns with False if not there
bool_imputer = SimpleImputer(strategy='constant', fill_value=False)

#flag categorical variables with missing if not there
cat_imputer = SimpleImputer(strategy='constant', fill_value='None')

num_cols = ["LivingArea", "YearBuilt"]
zero_cols = ["ParkingTotal", "BathroomsTotalInteger", "GarageSpaces", "AssociationFee"]
bool_cols = ["AttachedGarageYN", "PoolPrivateYN", "ViewYN"]
cat_cols = ["AssociationFeeFrequency"]

df["ParkingTotal"] = np.abs(df["ParkingTotal"])

for col in num_cols:
    df[col] = med_imputer.fit_transform(df[[col]])

for col in zero_cols:
    df[col] = zero_imputer.fit_transform(df[[col]])

for col in bool_cols:
    df[col] = bool_imputer.fit_transform(df[[col]])[:, 0]

#default is one story if not specified
df["Stories"] = one_imputer.fit_transform(df[["Stories"]])

for col in cat_cols:
    df[col] = cat_imputer.fit_transform(df[[col]])[:, 0]

# Encoding Variables

In [178]:
#encodes categorical variables
ohe_encoder = OneHotEncoder(drop = 'first', sparse_output=False)
cols_to_encode = ["AssociationFeeFrequency", "CountyOrParish", "StateOrProvince"]

transformer = ColumnTransformer(transformers=[("cat_encoder", ohe_encoder, cols_to_encode)],remainder="passthrough", verbose_feature_names_out=False)
transformer.set_output(transform="pandas")
df = transformer.fit_transform(df)

In [179]:
#adding column that checks if property has nearby school
df["has_elementary_school"] = df["ElementarySchool"].notna().astype(int)

# Train/Test Split

In [180]:
df['CloseDate'] = pd.to_datetime(df['CloseDate'])
df['CloseMonth'] = df['CloseDate'].dt.month
df['CloseYear'] = df['CloseDate'].dt.year

The latest month available in the dataset is May 2026. This will be the test set and the rest of the months are the training set.

In [181]:
test_month = 5
test_year = 2026

train_df = df[(df['CloseMonth'] < test_month) | (df['CloseYear'] < test_year)].copy()
test_df = df[(df['CloseMonth'] == test_month) & (df['CloseYear'] == test_year)].copy()

In [182]:
print(f"Shape of training set: {train_df.shape}")
print(f"Shape of test set: {test_df.shape}")

Shape of training set: (52357, 144)
Shape of test set: (12015, 144)


In [183]:
#export As CSV
train_df.to_csv('train_df.csv', index=False)
test_df.to_csv('test_df.csv',index=False)

files.download('train_df.csv')
files.download('test_df.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>